# LTX Financial Intelligence - Análise Financeira e KPIs

Este notebook realiza a análise financeira completa da **LTX Industrial** com base no histórico de 36 meses (2023 - 2025).

### Objetivos:
1. Processamento e Sanitização de Dados
2. Análise de Margens e Lucratividade (Receita, Margem Bruta, EBITDA)
3. Análise de Ciclo Operacional e Capital de Giro (DSO, DIO, DPO)
4. Análise de Fluxo de Caixa e CAPEX
5. Consolidação de KPIs para Tomada de Decisão Executiva


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configurações de estilo para visualização
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Carregar dados
file_path = '../data/financial_data.csv' if os.path.exists('../data/financial_data.csv') else 'financial_data.csv'
df = pd.read_csv(file_path)
df['date'] = pd.to_datetime(df['date'])
df.head()


## 1. Cálculo de KPIs e Margens


In [ ]:
# Margens Percentuais
df['gross_margin_%'] = (df['gross_profit'] / df['revenue']) * 100
df['ebitda_margin_%'] = (df['ebitda'] / df['revenue']) * 100
df['opex_ratio_%'] = (df['opex'] / df['revenue']) * 100

# Indicadores de Capital de Giro (Working Capital)
df['dso_days'] = (df['accounts_receivable'] / df['revenue']) * 30
df['dio_days'] = (df['inventory'] / df['cogs']) * 30
df['dpo_days'] = (df['accounts_payable'] / df['cogs']) * 30
df['cash_conversion_cycle'] = df['dso_days'] + df['dio_days'] - df['dpo_days']

df[['date', 'business_unit', 'revenue', 'gross_margin_%', 'ebitda_margin_%', 'cash_conversion_cycle']].head()


## 2. Resumo Consolidado por Unidade de Negócio


In [ ]:
summary_bu = df.groupby('business_unit').agg({
    'revenue': ['sum', 'mean'],
    'gross_profit': 'sum',
    'ebitda': 'sum',
    'gross_margin_%': 'mean',
    'ebitda_margin_%': 'mean',
    'operating_cash_flow': 'sum',
    'capex': 'sum'
}).round(2)

summary_bu


## 3. Visualização de Desempenho e Tendências


In [ ]:
# Evolução da Receita Mensal por Divisão
plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x='date', y='revenue', hue='business_unit', marker='o', linewidth=2.5)
plt.title('Evolução da Receita Mensal por Unidade de Negócio (2023 - 2025)', fontsize=14, fontweight='bold')
plt.ylabel('Receita (R$)', fontsize=12)
plt.xlabel('Data', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Margem EBITDA ao longo do tempo
plt.figure(figsize=(14, 6))
sns.lineplot(data=df, x='date', y='ebitda_margin_%', hue='business_unit', marker='s', linewidth=2)
plt.axhline(df['ebitda_margin_%'].mean(), color='red', linestyle='--', label='Média Geral EBITDA %')
plt.title('Evolução da Margem EBITDA (%)', fontsize=14, fontweight='bold')
plt.ylabel('Margem EBITDA (%)', fontsize=12)
plt.xlabel('Data', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Exportação dos KPIs Consolidados Anuais


In [ ]:
kpi_annual = df.groupby(['year']).agg({
    'revenue': 'sum',
    'gross_profit': 'sum',
    'ebitda': 'sum',
    'operating_cash_flow': 'sum',
    'capex': 'sum'
}).reset_index()

kpi_annual['gross_margin_%'] = (kpi_annual['gross_profit'] / kpi_annual['revenue']) * 100
kpi_annual['ebitda_margin_%'] = (kpi_annual['ebitda'] / kpi_annual['revenue']) * 100

print("Resumo Financeiro Anual (Consolidado):")
print(kpi_annual.round(2))
